# ResolveAI — QLoRA training on Google Colab

Fine-tunes `Qwen/Qwen2.5-3B-Instruct` with QLoRA on the ResolveAI complaint-classification dataset. Run top-to-bottom on any Colab GPU runtime.

## Before you start
1. Upload the `fine_tuning/` folder from your local repo to Google Drive at `MyDrive/resolveai/fine_tuning/` (drag-and-drop the whole folder into the Drive web UI).
2. **Runtime → Change runtime type → GPU**. Any tier works; the hardware-adaptive cell below picks the right dtype.
3. **Pro only:** Runtime → Manage sessions → enable **background execution** so training survives a closed tab.

**Expected wall time:** ~3h on T4, ~2h on L4, ~30-45 min on A100.  
**Expected cost (Pro):** 5-15 compute units out of 100 monthly.  
**Output:** ~70 MB LoRA adapter at `MyDrive/resolveai/checkpoints/resolveai-sentiment-lora/`.

In [ ]:
# 1. Confirm GPU runtime + see what we got
!nvidia-smi

## 2. Mount Google Drive

Checkpoints land on Drive so a Colab disconnection mid-training doesn't lose progress. With `save_strategy: steps / save_steps: 100`, you can resume from the most recent checkpoint by re-running this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Workspace + dependencies

Verifies your upload, creates the checkpoint/results directories on Drive, installs the QLoRA stack.

In [ ]:
import os

PROJECT = '/content/drive/MyDrive/resolveai'
FT_DIR = f'{PROJECT}/fine_tuning'
CHECKPOINT_DIR = f'{PROJECT}/checkpoints/resolveai-sentiment-lora'
RESULTS_DIR = f'{PROJECT}/results'
CFG_PATH = f'{FT_DIR}/configs/training_config.yaml'

assert os.path.isdir(FT_DIR), (
    f'Missing {FT_DIR}. Upload your local fine_tuning/ folder to Drive first.'
)
for required in ('data/formatted/train.jsonl', 'data/formatted/val.jsonl', 'data/formatted/test.jsonl'):
    p = f'{FT_DIR}/{required}'
    assert os.path.isfile(p), f'Missing {p}. Run 02_format_training_data.py locally and re-upload.'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

%pip install -q -U transformers peft trl bitsandbytes accelerate datasets pyyaml matplotlib

## 4. Hardware-adaptive config rewrite

T4 / V100 have **no bf16 tensor cores** so we must use fp16. L4 / A100 / H100 do bf16 natively — use it (larger exponent range, no GradScaler hassles). We also override `output_dir` so checkpoints write to Drive instead of ephemeral Colab disk.

Skip this cell if you've already customized the YAML manually.

In [ ]:
import yaml, subprocess

with open(CFG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['training']['output_dir'] = CHECKPOINT_DIR

gpu_name = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']
).decode().strip()
print(f'Detected GPU: {gpu_name}')

ampere_or_newer = any(arch in gpu_name for arch in ('A100', 'A10', 'L4', 'L40', 'H100'))
if ampere_or_newer:
    print('-> switching to bf16 (Ampere/Ada/Hopper)')
    cfg['training']['fp16'] = False
    cfg['training']['bf16'] = True
    cfg['quantization']['bnb_4bit_compute_dtype'] = 'bfloat16'
else:
    print('-> keeping fp16 (T4/V100 — no bf16 tensor cores)')
    cfg['training']['fp16'] = True
    cfg['training']['bf16'] = False
    cfg['quantization']['bnb_4bit_compute_dtype'] = 'float16'

with open(CFG_PATH, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print()
print('Final config:')
print(f"  output_dir:    {cfg['training']['output_dir']}")
print(f"  fp16 / bf16:   {cfg['training']['fp16']} / {cfg['training']['bf16']}")
print(f"  compute_dtype: {cfg['quantization']['bnb_4bit_compute_dtype']}")
print(f"  class_weights enabled: {cfg.get('class_weights', {}).get('enabled', False)}")

## 5. Smoke test the harness (no GPU work yet)

Validates the data path, reports the class-weight derivation on the real distribution, and confirms the JSONL records have the expected `messages` key. Takes ~5 seconds. If this fails, fix it before booking a 3-hour training run.

In [ ]:
!python {FT_DIR}/03_train_qlora.py --dry-run --config {CFG_PATH}

## 6. Train

**This is the long cell.** ~2.5-3h on T4, less on L4/A100. Progress logs every 10 steps (loss), eval every 100 steps (eval_loss). The trainer keeps the 3 most recent + best-by-eval-loss checkpoints (`save_total_limit: 3`) and reloads the best one at the end (`load_best_model_at_end: true`).

**If Colab disconnects:** re-run from this cell. TRL's `resume_from_checkpoint=True` will resume from the latest Drive-persisted checkpoint.

In [ ]:
!python {FT_DIR}/03_train_qlora.py --config {CFG_PATH}

## 7. Verify the saved adapter

Expect `adapter_config.json`, `adapter_model.safetensors`, the tokenizer files, and `training_args.bin`. Total ~70 MB.

In [ ]:
!ls -lh {CHECKPOINT_DIR}
!echo '---'
!du -sh {CHECKPOINT_DIR}

## 8. Evaluate — batched (adapter + baseline in one pass)

**Why not `04_evaluate.py`?** That script loads the base in 4-bit and
generates one example at a time (~24 s/sample → ~6.6 h for 995 on A100,
and it gets killed by Colab before finishing). This cell does it the fast
way: base in **bf16**, **batched** generation (batch=64), **left-padding**
(required for decoder-only batched generate), and `model.disable_adapter()`
so the 6 GB base loads **once** and serves both the fine-tuned and baseline
passes. ~0.5 s/sample → both passes in ~15-20 min.

Writes `metrics_adapter.json`, `metrics_baseline.json`, `preds_*.json`,
and `eval_summary.md` to `RESULTS_DIR`. Invalid JSON is scored as **wrong**
(sentinel label), not dropped — for a structured-output classifier a
malformed response IS a failure.


In [ ]:
# peft on Colab can clash with a stale torchao — remove it so the
# version check short-circuits (we don't use torchao). Does NOT survive a
# runtime restart; re-run this if you restart the session.
!pip uninstall -y torchao -q

import json, re, os, time, contextlib, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sklearn.metrics import classification_report, f1_score
from scipy.stats import spearmanr

ADAPTER   = CHECKPOINT_DIR
TEST_PATH = f"{FT_DIR}/data/formatted/test.jsonl"
RESULTS   = RESULTS_DIR
BASE      = "Qwen/Qwen2.5-3B-Instruct"
LIMIT     = None      # None = all 995; set an int for a quick pass
BATCH     = 64        # 40GB A100 handles this; drop to 32 if OOM
MAX_NEW   = 384
os.makedirs(RESULTS, exist_ok=True)

# tokenizer FROM THE ADAPTER DIR (carries the exact trained chat template)
tok = AutoTokenizer.from_pretrained(ADAPTER, padding_side="left")
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# base in bf16, loaded ONCE; attach LoRA. baseline = disable_adapter()
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16, device_map="cuda")
model = PeftModel.from_pretrained(model, ADAPTER).eval()

rows = [json.loads(l) for l in open(TEST_PATH)]
if LIMIT: rows = rows[:LIMIT]
prompts, golds = [], []
for r in rows:
    msgs = r["messages"]
    golds.append(json.loads(next(m for m in msgs if m["role"] == "assistant")["content"]))
    ctx = [m for m in msgs if m["role"] != "assistant"]
    prompts.append(tok.apply_chat_template(ctx, tokenize=False, add_generation_prompt=True))

def parse(txt):
    m = re.search(r"\{.*\}", txt, re.DOTALL)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception: return None

def lab(p, key):
    if not p: return "INVALID"
    v = p.get(key)
    return v if (isinstance(v, str) and v) else "INVALID"

def urg(p):
    if not p: return None
    try: return int(p.get("urgency"))
    except (TypeError, ValueError): return None

def run_pass(label, disable_adapter):
    cm = model.disable_adapter() if disable_adapter else contextlib.nullcontext()
    preds, t0 = [], time.time()
    with cm:
        for i in range(0, len(prompts), BATCH):
            enc = tok(prompts[i:i+BATCH], return_tensors="pt", padding=True,
                      truncation=True, max_length=2048).to("cuda")
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                                     pad_token_id=tok.pad_token_id)
            new = out[:, enc["input_ids"].shape[1]:]
            preds += [parse(t) for t in tok.batch_decode(new, skip_special_tokens=True)]
            print("  [%s] %d/%d (%.0fs)" % (label, min(i+BATCH, len(prompts)),
                                            len(prompts), time.time()-t0))
    json.dump(preds, open("%s/preds_%s.json" % (RESULTS, label), "w"))

    sg = [g["sentiment"] for g in golds]; sp = [lab(p, "sentiment") for p in preds]
    ig = [g["intent"]    for g in golds]; ip = [lab(p, "intent")    for p in preds]
    nv = sum(p is not None for p in preds)
    pairs = [(int(g["urgency"]), urg(p)) for p, g in zip(preds, golds) if urg(p) is not None]
    if pairs:
        ug, up = zip(*pairs)
        mae = sum(abs(a-b) for a, b in zip(ug, up)) / len(ug)
        rho = spearmanr(ug, up).correlation if len(set(ug)) > 1 else float("nan")
    else:
        mae, rho = float("nan"), float("nan")

    real_sent = ["neutral", "negative", "extreme_negative"]
    real_int  = sorted({g["intent"] for g in golds})
    rep = {"label": label, "n": len(preds), "valid_json": nv,
           "valid_json_pct": round(100*nv/len(preds), 2),
           # macro-F1 over REAL classes only (exclude the INVALID sentinel)
           "sentiment_macro_f1": round(f1_score(sg, sp, labels=real_sent, average="macro", zero_division=0), 4),
           "intent_macro_f1":    round(f1_score(ig, ip, labels=real_int,  average="macro", zero_division=0), 4),
           "urgency_mae": round(mae, 3),
           "urgency_spearman": (round(rho, 3) if rho == rho else None),
           "sentiment_text": classification_report(sg, sp, zero_division=0),
           "intent_text":    classification_report(ig, ip, zero_division=0)}
    json.dump(rep, open("%s/metrics_%s.json" % (RESULTS, label), "w"), indent=2)
    return rep

print("=== ADAPTER PASS ===");  adp = run_pass("adapter",  False)
print("=== BASELINE PASS ==="); bsl = run_pass("baseline", True)

def block(r):
    L = ["valid JSON: %d/%d (%s%%)" % (r["valid_json"], r["n"], r["valid_json_pct"]),
         "sentiment macro-F1: %s   intent macro-F1: %s" % (r["sentiment_macro_f1"], r["intent_macro_f1"]),
         "urgency MAE: %s   Spearman: %s" % (r["urgency_mae"], r["urgency_spearman"]),
         "", "-- SENTIMENT --", r["sentiment_text"], "-- INTENT --", r["intent_text"]]
    return "\n".join(L)

s = ["# ResolveAI - fine-tune vs baseline (bf16, full test set)", ""]
s.append("sentiment macro-F1: baseline %s -> adapter %s (delta %+.4f)" % (
    bsl["sentiment_macro_f1"], adp["sentiment_macro_f1"], adp["sentiment_macro_f1"]-bsl["sentiment_macro_f1"]))
s.append("intent macro-F1: baseline %s -> adapter %s (delta %+.4f)" % (
    bsl["intent_macro_f1"], adp["intent_macro_f1"], adp["intent_macro_f1"]-bsl["intent_macro_f1"]))
s.append("valid JSON: baseline %s%% -> adapter %s%%" % (bsl["valid_json_pct"], adp["valid_json_pct"]))
s.append("urgency MAE: baseline %s -> adapter %s" % (bsl["urgency_mae"], adp["urgency_mae"]))
s += ["", "## ADAPTER", block(adp), "", "## BASELINE", block(bsl)]
summary = "\n".join(s)
open("%s/eval_summary.md" % RESULTS, "w").write(summary)
print("\n" + summary)


## 9. Export to GGUF Q4_K_M (run here, not on an 8 GB Mac)

Merge the LoRA into the base and quantize to Q4_K_M **on Colab**, where
the 6 GB base is already resident in VRAM (instant merge, no reload). The
local M3 only has 8 GB, so doing the merge there thrashes swap. Writes the
~2 GB GGUF + a Modelfile (training-time system prompt + ChatML template +
deterministic params) to Drive, then downloads both.

Locally afterwards: drop both files in `fine_tuning/models/`, then
`ollama create resolveai-sentiment -f Modelfile`.


In [ ]:
import os, subprocess, torch

MERGED   = "/content/merged_fp16"
GGUF_DIR = f"{PROJECT}/gguf"
os.makedirs(GGUF_DIR, exist_ok=True)

# 1) merge LoRA into base — reuses the model already in RAM, no reload
merged = model.merge_and_unload()
merged.save_pretrained(MERGED, safe_serialization=True)
tok.save_pretrained(MERGED)
del merged; torch.cuda.empty_cache()
print("merged ->", MERGED)

# 2) llama.cpp: clone + build the quantizer (~3-4 min)
if not os.path.isdir("/content/llama.cpp"):
    subprocess.run("git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp",
                   shell=True, check=True)
subprocess.run("pip install -q -r /content/llama.cpp/requirements.txt", shell=True, check=True)
subprocess.run("cmake -B /content/llama.cpp/build /content/llama.cpp -DLLAMA_CURL=OFF "
               "&& cmake --build /content/llama.cpp/build --target llama-quantize -j",
               shell=True, check=True)

# 3) convert merged HF -> fp16 GGUF -> Q4_K_M (Q4 persisted to Drive)
F16 = "/content/resolveai-f16.gguf"
Q4  = GGUF_DIR + "/resolveai-sentiment-q4_k_m.gguf"
subprocess.run("python /content/llama.cpp/convert_hf_to_gguf.py %s --outfile %s --outtype f16"
               % (MERGED, F16), shell=True, check=True)
subprocess.run("/content/llama.cpp/build/bin/llama-quantize %s %s Q4_K_M" % (F16, Q4),
               shell=True, check=True)
print("Q4_K_M: %.0f MB" % (os.path.getsize(Q4)/1e6))

# 4) Modelfile (no f-strings so Ollama's {{ }} template braces survive)
SYS = ("You are a financial complaint classifier. "
       "Analyze the complaint and output a structured JSON classification.")
template = ('TEMPLATE """{{ if .System }}<|im_start|>system\n'
            '{{ .System }}<|im_end|>\n{{ end }}<|im_start|>user\n'
            '{{ .Prompt }}<|im_end|>\n<|im_start|>assistant\n"""')
modelfile = ("FROM ./resolveai-sentiment-q4_k_m.gguf\n" + template + "\n"
             + 'SYSTEM """' + SYS + '"""\n'
             + "PARAMETER temperature 0\nPARAMETER top_p 1\n"
             + 'PARAMETER stop "<|im_end|>"\n')
open(GGUF_DIR + "/Modelfile", "w").write(modelfile)
print("wrote Modelfile ->", GGUF_DIR)

# 5) download both
from google.colab import files
files.download(Q4)
files.download(GGUF_DIR + "/Modelfile")


## 10. Read the eval summary

`eval_summary.md` has the headline adapter-vs-baseline deltas plus the
full per-class reports. **Reconciliation note:** the macro-F1 in this
summary is computed over the *real* classes only (`labels=` excludes the
`INVALID` sentinel) — earlier hand-rolled cells that averaged the
zero-support sentinel row reported a deflated ~0.63 instead of the true
~0.84.


In [ ]:
path = f'{RESULTS_DIR}/eval_summary.md'
if os.path.isfile(path):
    print(open(path).read())
else:
    print(f'(missing: {path} — run the eval cell first)')


## 11. Bring it all back to your local repo

The eval cell already downloaded nothing; the GGUF cell downloaded the
GGUF + Modelfile. To pull everything (adapter + results + gguf) into
`Resolve_AI/fine_tuning/` (all gitignored), either use **Google Drive for
Desktop** and `cp -R` from
`~/Library/CloudStorage/GoogleDrive-<email>/My Drive/resolveai/...`, or
run the zip cell below.

**Gotchas learned the hard way:**
- Downloading a *folder* from the Drive web UI gives a `*-NNN.zip`, not
  loose files — `unzip -l` it before assuming a file is missing.
- The GGUF goes next to the Modelfile (`FROM ./resolveai-sentiment-q4_k_m.gguf`
  is relative). Then: `ollama create resolveai-sentiment -f Modelfile`.


In [ ]:
# Option B — zip + browser download
import shutil, os
from google.colab import files

adapter_zip = '/content/resolveai-sentiment-lora.zip'
shutil.make_archive(adapter_zip[:-4], 'zip', CHECKPOINT_DIR)
print(f'adapter zip: {os.path.getsize(adapter_zip) / 1e6:.1f} MB')

results_zip = '/content/results.zip'
shutil.make_archive(results_zip[:-4], 'zip', RESULTS_DIR)
print(f'results zip: {os.path.getsize(results_zip) / 1e6:.2f} MB')

files.download(adapter_zip)
files.download(results_zip)

## Done

Once the adapter + results are in your local repo under `Resolve_AI/fine_tuning/`, next steps are:

1. `05_export_gguf.py` — merge LoRA into base, convert to GGUF Q4_K_M, write a Modelfile for Ollama.
2. `ollama create resolveai-sentiment -f Modelfile` and `ollama run` smoke test.
3. Day 13 backend integration — `classifier.py`, `llm_client.py`, `llmops_tracker.py`, `classification_worker.py`.

Both directories are gitignored, so they won’t accidentally land in commits.